In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
import json
import holidays

In [5]:
RAW_PATH   = "Crimes_2001.csv"
OUT_DIR    = Path("processed")
OUT_DIR.mkdir(exist_ok=True)

YEAR_START, YEAR_END = 2016, 2025
TOP_N_TYPES  = 5      # 只保留频率最高的N种犯罪类型
LOOKBACK     = 180    # 30天 × 6槽
SLOTS_PER_DAY = 6

# Chicago bounding box
LAT_MIN, LAT_MAX = 41.644, 42.023
LON_MIN, LON_MAX = -87.940, -87.524

# 1 km in degrees
GRID_LAT = 1.0 / 111.0
GRID_LON = 1.0 / (111.0 * np.cos(np.radians(41.83)))

# ── 1. Load & basic filter ─────────────────────────────────────────────────────
print("Loading...")
usecols = ["Date", "Primary Type", "Latitude", "Longitude"]
df = pd.read_csv(RAW_PATH, usecols=usecols)
df["Date"] = pd.to_datetime(df["Date"], format="%m/%d/%Y %I:%M:%S %p")

df = df[(df["Date"].dt.year >= YEAR_START) & (df["Date"].dt.year <= YEAR_END)]
df = df[~((df["Date"].dt.month == 1) & (df["Date"].dt.day == 1))]   # 剔除1月1日伪影
df = df.dropna(subset=["Latitude", "Longitude"])
df = df[
    (df["Latitude"]  >= LAT_MIN) & (df["Latitude"]  <= LAT_MAX) &
    (df["Longitude"] >= LON_MIN) & (df["Longitude"] <= LON_MAX)
]
print(f"Records after filter: {len(df):,}")

# ── 2. Time slot & temporal features ──────────────────────────────────────────
def hour_to_slot(hour):
    if hour >= 22 or hour < 2:  return 0   # 22-02 深夜
    elif hour < 6:              return 1   # 02-06 凌晨
    elif hour < 10:             return 2   # 06-10 上午
    elif hour < 14:             return 3   # 10-14 午间
    elif hour < 18:             return 4   # 14-18 下午
    else:                       return 5   # 18-22 傍晚

df["slot"] = df["Date"].dt.hour.map(hour_to_slot)

# slot 0 跨越午夜：00:00-01:59 属于"前一天的深夜"，日期往前移一天
# 这样 Jan 5 22:30 和 Jan 6 01:30 都会归入 Jan 5 的 slot 0
df["date"] = df["Date"].dt.normalize()
mask_early = df["Date"].dt.hour < 2          # 00:00-01:59（slot 0 前半段）
df.loc[mask_early, "date"] -= pd.Timedelta(days=1)

# 时间特征从修正后的 date 推导（保证 weekday/month/节假日与 slot 语义一致）
df["weekday"]    = df["date"].dt.dayofweek      # 0=Mon … 6=Sun
df["month"]      = df["date"].dt.month          # 1–12
df["is_weekend"] = df["weekday"].isin([5, 6]).astype(np.int8)

il_holidays = holidays.US(state="IL", years=range(YEAR_START, YEAR_END + 1))
df["is_holiday"] = df["date"].dt.date.map(lambda d: int(d in il_holidays)).astype(np.int8)

# 循环编码
df["slot_sin"]    = np.sin(2 * np.pi * df["slot"]    / SLOTS_PER_DAY)
df["slot_cos"]    = np.cos(2 * np.pi * df["slot"]    / SLOTS_PER_DAY)
df["weekday_sin"] = np.sin(2 * np.pi * df["weekday"] / 7)
df["weekday_cos"] = np.cos(2 * np.pi * df["weekday"] / 7)
df["month_sin"]   = np.sin(2 * np.pi * df["month"]   / 12)
df["month_cos"]   = np.cos(2 * np.pi * df["month"]   / 12)

# ── 3. Spatial grid ────────────────────────────────────────────────────────────
df["grid_row"] = ((df["Latitude"]  - LAT_MIN) / GRID_LAT).astype(int)
df["grid_col"] = ((df["Longitude"] - LON_MIN) / GRID_LON).astype(int)

N_ROWS = int((LAT_MAX - LAT_MIN) / GRID_LAT) + 1
N_COLS = int((LON_MAX - LON_MIN) / GRID_LON) + 1
df["grid_id"] = df["grid_row"] * N_COLS + df["grid_col"]
print(f"Grid size: {N_ROWS} rows × {N_COLS} cols = {N_ROWS * N_COLS} cells")

# ── 4. Crime type filter ───────────────────────────────────────────────────────
top_types = df["Primary Type"].value_counts().head(TOP_N_TYPES).index.tolist()
df = df[df["Primary Type"].isin(top_types)]
crime_type_map = {t: i for i, t in enumerate(top_types)}
df["crime_type"] = df["Primary Type"].map(crime_type_map)
print(f"Crime types kept: {top_types}")

# ── 5. 全局时间步索引 ──────────────────────────────────────────────────────────
# date_to_idx 从 YEAR_START-01-01（含）往前一天开始，以覆盖被移位到前一天的记录
# 实际起点仍是 YEAR_START-01-02（1月1日过滤后的第一天），
# 但 slot 0 前半段移位后可能落到 YEAR_START-01-01，故索引从该日构建
all_dates = pd.date_range(start=f"{YEAR_START}-01-01", end=f"{YEAR_END}-12-31", freq="D")
# 剔除每年 1月1日（数据伪影），移位后落到 1月1日的记录同样会被 dropna 过滤
all_dates = all_dates[~((all_dates.month == 1) & (all_dates.day == 1))]

date_to_idx = {d.normalize(): i for i, d in enumerate(all_dates)}
N_DAYS  = len(all_dates)
N_STEPS = N_DAYS * SLOTS_PER_DAY

df["day_idx"]   = df["date"].map(date_to_idx)
df["time_step"] = df["day_idx"] * SLOTS_PER_DAY + df["slot"]
# dropna 会自动过滤：① 日期不在索引内（如落到被删除的1月1日）② 其他边界情况
df = df.dropna(subset=["day_idx"])
df["time_step"] = df["time_step"].astype(int)

# ── 6. 聚合：count per (time_step, grid_id, crime_type) ───────────────────────
counts = (
    df.groupby(["time_step", "grid_id", "crime_type"])
      .size().reset_index(name="count")
)

# ── 7. 构建密集张量 (N_STEPS, N_GRIDS, N_TYPES) ────────────────────────────────
active_grids = sorted(df["grid_id"].unique())
grid_remap   = {g: i for i, g in enumerate(active_grids)}
N_GRIDS = len(active_grids)
N_TYPES = len(top_types)
print(f"Active grids: {N_GRIDS}  |  Time steps: {N_STEPS}  |  Crime types: {N_TYPES}")

tensor = np.zeros((N_STEPS, N_GRIDS, N_TYPES), dtype=np.int16)
counts["grid_idx"] = counts["grid_id"].map(grid_remap)
counts = counts.dropna(subset=["grid_idx"])
counts["grid_idx"] = counts["grid_idx"].astype(int)
tensor[counts["time_step"].values, counts["grid_idx"].values, counts["crime_type"].values] = counts["count"].values

print(f"Tensor shape: {tensor.shape}  (~{tensor.nbytes/1e6:.0f} MB)")
print(f"Non-zero entries: {np.count_nonzero(tensor):,}  ({100*np.count_nonzero(tensor)/tensor.size:.2f}%)")

# ── 8. 时序切分索引（8:1:1） ──────────────────────────────────────────────────
n_predictable = N_STEPS - LOOKBACK
train_end = LOOKBACK + int(n_predictable * 0.8)
val_end   = LOOKBACK + int(n_predictable * 0.9)
print(f"\nSplit time_steps:  train [0, {train_end})  |  val [{train_end}, {val_end})  |  test [{val_end}, {N_STEPS})")
print(f"Predictable windows:  train {train_end-LOOKBACK}  |  val {val_end-train_end}  |  test {N_STEPS-val_end}")

# ── 9. 时间特征表 time_features (N_STEPS, 8) ─────────────────────────────────
steps    = np.arange(N_STEPS)
day_idxs = steps // SLOTS_PER_DAY
slots    = steps  % SLOTS_PER_DAY
dates_arr = all_dates[day_idxs]

weekdays   = dates_arr.dayofweek.values
months     = dates_arr.month.values
is_weekend = (weekdays >= 5).astype(np.float32)
is_hol     = np.array([int(d.date() in il_holidays) for d in dates_arr], dtype=np.float32)

time_features = np.column_stack([
    np.sin(2 * np.pi * slots    / SLOTS_PER_DAY),
    np.cos(2 * np.pi * slots    / SLOTS_PER_DAY),
    np.sin(2 * np.pi * weekdays / 7),
    np.cos(2 * np.pi * weekdays / 7),
    np.sin(2 * np.pi * months   / 12),
    np.cos(2 * np.pi * months   / 12),
    is_weekend,
    is_hol,
]).astype(np.float32)
print(f"\ntime_features shape: {time_features.shape}")

# ── 10. 保存 ──────────────────────────────────────────────────────────────────
np.save(OUT_DIR / "tensor.npy",        tensor)
np.save(OUT_DIR / "time_features.npy", time_features)

meta = {
    "n_rows": int(N_ROWS), "n_cols": int(N_COLS),
    "n_grids": int(N_GRIDS), "n_types": int(N_TYPES),
    "n_steps": int(N_STEPS), "lookback": int(LOOKBACK),
    "train_end": int(train_end), "val_end": int(val_end),
    "active_grids": [int(g) for g in active_grids],
    "grid_remap": {str(k): int(v) for k, v in grid_remap.items()},
    "crime_types": top_types,
    "crime_type_map": {k: int(v) for k, v in crime_type_map.items()},
    "date_to_idx": {str(k): int(v) for k, v in date_to_idx.items()},
    "lat_min": float(LAT_MIN), "lon_min": float(LON_MIN),
    "grid_lat": float(GRID_LAT), "grid_lon": float(GRID_LON),
    "time_feature_cols": ["slot_sin","slot_cos","weekday_sin","weekday_cos",
                          "month_sin","month_cos","is_weekend","is_holiday"],
}
with open(OUT_DIR / "meta.json", "w") as f:
    json.dump(meta, f, indent=2)

# ── 11. 保存事件级别数据（FlexiCrime / XGBoost 事件级模型） ──────────────────
event_cols = [
    "time_step", "slot", "weekday", "month", "is_weekend", "is_holiday",
    "slot_sin", "slot_cos", "weekday_sin", "weekday_cos", "month_sin", "month_cos",
    "Latitude", "Longitude", "grid_row", "grid_col", "grid_id",
    "crime_type", "Primary Type",
]

df_train = df[df["time_step"] <  train_end].reset_index(drop=True)
df_val   = df[(df["time_step"] >= train_end) & (df["time_step"] < val_end)].reset_index(drop=True)
df_test  = df[df["time_step"] >= val_end].reset_index(drop=True)

df_train[event_cols].to_parquet(OUT_DIR / "events_train.parquet", index=False)
df_val[event_cols].to_parquet(OUT_DIR   / "events_val.parquet",   index=False)
df_test[event_cols].to_parquet(OUT_DIR  / "events_test.parquet",  index=False)

print(f"\nEvents  Train: {len(df_train):,}  Val: {len(df_val):,}  Test: {len(df_test):,}")
print(f"\nFiles saved to {OUT_DIR}:")
for f in sorted(OUT_DIR.iterdir()):
    print(f"  {f.name:30s}  {f.stat().st_size/1e6:7.1f} MB")


Loading...
Records after filter: 2,446,558
Grid size: 43 rows × 35 cols = 1505 cells
Crime types kept: ['THEFT', 'BATTERY', 'CRIMINAL DAMAGE', 'ASSAULT', 'DECEPTIVE PRACTICE']
Active grids: 674  |  Time steps: 21858  |  Crime types: 5
Tensor shape: (21858, 674, 5)  (~147 MB)
Non-zero entries: 1,548,766  (2.10%)

Split time_steps:  train [0, 17522)  |  val [17522, 19690)  |  test [19690, 21858)
Predictable windows:  train 17342  |  val 2168  |  test 2168

time_features shape: (21858, 8)

Events  Train: 1,321,622  Val: 171,614  Test: 158,103

Files saved to processed:
  events_test.parquet                 2.5 MB
  events_train.parquet               23.5 MB
  events_val.parquet                  2.7 MB
  meta.json                           0.1 MB
  tensor.npy                        147.3 MB
  time_features.npy                   0.7 MB
